# Teams Analysis

Loads the `teams` table written by `teams_etl.py` and provides basic profiling.

In [ ]:
import duckdb, os, glob, json, pandas as pd, plotly.express as px

In [ ]:
# Resolve DuckDB database (pick latest mydb*.duckdb if not specified)
db_candidates = sorted(glob.glob('../../data/mydb2024-25*.duckdb'))
db_path = db_candidates[-1] if db_candidates else '../data/mydb.duckdb'
print(f'Using DuckDB database: {db_path}')
con = duckdb.connect(db_path)

In [ ]:
# Load teams table
try:
    df_teams = con.execute('SELECT * FROM teams').df()
except Exception as e:
    raise RuntimeError(f'Teams table not found: {e}')
df_teams.head()

In [ ]:
# Basic stats
print(f'Total teams: {len(df_teams)}')
display(df_teams.describe(include='all').transpose())

In [ ]:
# Visual: codeLocal / codeLatin uniqueness
cols = [c for c in ['codeLocal','codeLatin'] if c in df_teams.columns]
long = df_teams.melt(value_vars=cols, var_name='code_type', value_name='code').dropna()
px.bar(long.groupby('code_type')['code'].nunique().reset_index(), x='code_type', y='code', title='Unique Codes per Type')